In [1]:
import torch
import pandas as pd
import torch.nn as nn
from src.configuration.config import set_seed, SEED

In [2]:
set_seed(SEED)

In [3]:
df = pd.read_csv("../data/processed/combined.csv")

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameter Tuning
### Testing different Configs with Coordinate Search based on AU02 Outer Brow Raiser

In [5]:
def sequential_hyperparameter_search(
    df,
    test_au,
    base_config,
    search_space,
    sources,
    device
):
    best_config = base_config.copy()
    all_results = []

    for param_name, values in search_space.items():
        print("\n" + "=" * 50)
        print(f"TESTING HYPERPARAMETER: {param_name}")
        print("=" * 50)

        step_results = []

        for value in values:
            config = best_config.copy()

            # Tuning window_size and stride together
            if param_name == "window_config":
                config.update(value)
            else:
                config[param_name] = value

            print(f"\nTesting {param_name} = {value}")
            print("Current config:")
            print(config)

            fold_result = LOSO_loop_config(
                df=df,
                test_au=test_au,
                config=config,
                param_name=param_name,
                param_value=value,
                sources=sources,
                device=device,
                seed=SEED
            )

            step_results.append(fold_result)
            all_results.append(fold_result)
            
        step_results_df = pd.DataFrame(step_results)

        step_results_df["param_value_group"] = (
            step_results_df["param_value"].astype(str)
        )
        
        mean_results = (
            step_results_df
            .groupby("param_value_group", as_index=False)["bal. accuracy"]
            .mean()
        )
        
        best_row = mean_results.loc[
            mean_results["bal. accuracy"].idxmax()
        ]
        
        best_param_value = next(
            result["param_value"]
            for result in step_results
            if str(result["param_value"]) == best_row["param_value_group"]
        )
        
        if param_name == "window_config":
            best_config.update(best_param_value)
        else:
            best_config[param_name] = best_param_value
        
        print("\nBest result in current step:")
        print(f"Parameter: {param_name}")
        print(f"Value: {best_param_value}")
        print(f"Mean Loss: {best_row['bal. accuracy']:.4f}")

    all_results = pd.DataFrame(all_results)
    return best_config, all_results

In [6]:
from src.training.evaluation import full_test_evaluation_per_split
from src.training.training import full_training_per_split
from src.MILArchitecture.AttentionMIL import AttentionMIL
from src.training.loaders import initialize_loaders_per_split
from src.MILArchitecture.windows import initialize_all_bags
from sklearn.metrics import balanced_accuracy_score, classification_report


def LOSO_loop_config(
    df,
    test_au,
    config,
    param_name,
    param_value,
    sources,
    device,
    input_dim=8,
    seed=42
):
    bags, labels, clip_ids, source_ids, metadata = initialize_all_bags(
        df,
        test_au,
        config
    )
    
    avg_loss = 0.0
    all_y_true = []
    all_y_pred = []

    for fold_idx, test_source in enumerate(sources):
        fold_seed = seed + fold_idx
        set_seed(fold_seed)

        train_loader, val_loader, test_loader = initialize_loaders_per_split(
            test_source=test_source,
            sources=sources,
            metadata=metadata,
            bags=bags,
            labels=labels,
            clip_ids=clip_ids,
            config=config,
            seed=fold_seed
        )

        model = AttentionMIL(input_dim=input_dim, dropout=config["dropout"]).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config["learning_rate"],
            weight_decay=config["weight_decay"]
        )

        criterion = nn.BCEWithLogitsLoss()

        best_model_state = full_training_per_split(
            num_epochs=config["num_epochs"],
            model=model,
            optimizer=optimizer,
            criterion=criterion,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            config=config,
            log=False
        )

        test_avg_loss, test_all_y_true, test_all_y_pred, _ = full_test_evaluation_per_split(
            best_model_state=best_model_state,
            model=model,
            criterion=criterion,
            test_loader=test_loader,
            test_source=test_source,
            device=device,
            config=config
        )
        
        avg_loss += test_avg_loss
        all_y_true.extend(test_all_y_true)
        all_y_pred.extend(test_all_y_pred)

    acc = balanced_accuracy_score(y_true=all_y_true, y_pred=all_y_pred)
    loss = avg_loss / len(sources)
    cr = classification_report(
        all_y_true,
        all_y_pred,
        labels=[0, 1],
        output_dict=True,
        zero_division=0
    )
    
    result = ({
        "param_name": param_name,
        "param_value": param_value,
        "loss": loss,
        "bal. accuracy": acc,
        "precision_0": cr["0"]["precision"],
        "precision_1": cr["1"]["precision"],
        "recall_0": cr["0"]["recall"],
        "recall_1": cr["1"]["recall"],
        "f1_0": cr["0"]["f1-score"],
        "f1_1": cr["1"]["f1-score"]
    })

    return result

In [7]:
base_config = {
    # Window generation: Strongly correlated (!)
    "window_size": 20,
    "stride": 10,

    # Optimization
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 16,
    "dropout": 0.2,

    # Training and evaluation (Not tuned)
    "num_epochs": 25,
    "threshold": 0.5
}

search_space = {
    "window_config": [
        {"window_size": 10, "stride": 5},
        {"window_size": 10, "stride": 10},
        {"window_size": 20, "stride": 10},
        {"window_size": 20, "stride": 20},
        {"window_size": 30, "stride": 15},
        {"window_size": 30, "stride": 30}
    ],

    "learning_rate": [
        1e-4,
        3e-4,
        1e-3,
        3e-3,
        1e-2,
        3e-2
    ],

    "weight_decay": [
        0.0,
        1e-5,
        1e-4,
        1e-3,
        1e-2
    ],

    "batch_size": [
        8,
        16,
        32,
        64
    ],

    "dropout": [
        0.0,
        0.1,
        0.2,
        0.3,
        0.4,
        0.5
    ]
}

In [8]:
from src.utils.data import sources, au_cols

best_config, summary = sequential_hyperparameter_search(
    df=df,
    test_au=au_cols,
    base_config=base_config,
    search_space=search_space,
    sources=sources,
    device=device
)


TESTING HYPERPARAMETER: window_config

Testing window_config = {'window_size': 10, 'stride': 5}
Current config:
{'window_size': 10, 'stride': 5, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 16, 'dropout': 0.2, 'num_epochs': 25, 'threshold': 0.5}

Testing window_config = {'window_size': 10, 'stride': 10}
Current config:
{'window_size': 10, 'stride': 10, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 16, 'dropout': 0.2, 'num_epochs': 25, 'threshold': 0.5}

Testing window_config = {'window_size': 20, 'stride': 10}
Current config:
{'window_size': 20, 'stride': 10, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 16, 'dropout': 0.2, 'num_epochs': 25, 'threshold': 0.5}

Testing window_config = {'window_size': 20, 'stride': 20}
Current config:
{'window_size': 20, 'stride': 20, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 16, 'dropout': 0.2, 'num_epochs': 25, 'threshold': 0.5}

Testing window_config = {'window_size': 30, 'stride': 

In [9]:
best_config

{'window_size': 10,
 'stride': 5,
 'learning_rate': 0.003,
 'weight_decay': 0.001,
 'batch_size': 16,
 'dropout': 0.2,
 'num_epochs': 25,
 'threshold': 0.5}

In [10]:
summary["param_value_group"] = summary["param_value"].apply(
    lambda x: str(x) if isinstance(x, dict) else x
)

summary = (
    summary
    .groupby(["param_name", "param_value_group"],
             as_index=False,
             sort=False)
    .mean(numeric_only=True)
)

In [11]:

summary["macro_f1"] = (summary["f1_0"] + summary["f1_1"]) / 2

Config is chosen by max. Balanced Accuracy instead of min. Loss -> Min. the loss often leads to prioritizing class 1. Choosing Balanced Accuracy increases the precision_0, recall_0 and f1_0 significantly in later tested hyperparameters, while performance for 1 does not drastically change. BUT Loss increases

In [12]:
summary

,param_name,param_value_group,loss,bal. accuracy,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,macro_f1
0,window_config,"{'window_size': 10, 'stride': 5}",0.723799,0.544907,0.485714,0.626263,0.314815,0.7750,0.382022,0.692737,0.537380
1,window_config,"{'window_size': 10, 'stride': 10}",0.703012,0.532639,0.468750,0.617647,0.277778,0.7875,0.348837,0.692308,0.520572
2,window_config,"{'window_size': 20, 'stride': 10}",0.737686,0.511343,0.434783,0.603604,0.185185,0.8375,0.259740,0.701571,0.480655
3,window_config,"{'window_size': 20, 'stride': 20}",0.724828,0.498843,0.400000,0.596330,0.185185,0.8125,0.253165,0.687831,0.470498
4,window_config,"{'window_size': 30, 'stride': 15}",0.720652,0.471065,0.318182,0.580357,0.129630,0.8125,0.184211,0.677083,0.430647
5,window_config,"{'window_size': 30, 'stride': 30}",0.733614,0.517361,0.444444,0.607477,0.222222,0.8125,0.296296,0.695187,0.495742
6,learning_rate,0.0001,0.683831,0.454861,0.324324,0.567010,0.222222,0.6875,0.263736,0.621469,0.442603
7,learning_rate,0.0003,0.666366,0.508102,0.423077,0.601852,0.203704,0.8125,0.275000,0.691489,0.483245
8,learning_rate,0.001,0.723799,0.544907,0.485714,0.626263,0.314815,0.7750,0.382022,0.692737,0.537380
9,learning_rate,0.003,0.794119,0.569444,0.512195,0.645161,0.388889,0.7500,0.442105,0.693642,0.567873


In [14]:
summary.to_latex(
    "../results/hyperparameter_config_BalAcc.tex",
    index=False,
    float_format="%.3f"
)